# Churn Prediction | Pré-Processamento
**Objetivo:** Transformar os dados brutos em input pronto para modelagem.  
**Etapas:** Limpeza → Engenharia de Features → Encoding/Scaling → Pipeline → Salvar datasets tratados.

## 0. Imports

In [11]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
import joblib

## 1. Limpeza

In [12]:
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Todos os 11 nulos tem tenure = 0: clientes que nunca geraram cobrança total
# Iremos imputar com mediana por grupo de Contract para preservar a distribuição
print(df[df['TotalCharges'].isnull()][['tenure', 'Contract', 'MonthlyCharges']])

df['TotalCharges'] = df.groupby('Contract')['TotalCharges']\
    .transform(lambda x: x.fillna(x.median()))

print(f"Nulos restantes: {df['TotalCharges'].isnull().sum()}")

      tenure  Contract  MonthlyCharges
488        0  Two year           52.55
753        0  Two year           20.25
936        0  Two year           80.85
1082       0  Two year           25.75
1340       0  Two year           56.05
3331       0  Two year           19.85
3826       0  Two year           25.35
4380       0  Two year           20.00
5218       0  One year           19.70
6670       0  Two year           73.35
6754       0  Two year           61.90
Nulos restantes: 0


In [13]:
# customerID é identificador único e não tem valor preditivo, então podemos removê-lo
df = df.drop(columns=['customerID'])

# Target binário: Yes -> 1, No -> 0
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print(f"Shape após limpeza: {df.shape}")
print(f"Target: {df['Churn'].value_counts().to_dict()}")

Shape após limpeza: (7043, 20)
Target: {0: 5174, 1: 1869}


In [14]:
# SeniorCitizen lido como int (0/1) iremos converter para categórico Yes/No
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

# Colunas de serviço têm 3 valores: Yes / No / No internet service
# "No internet service" é semanticamente equivalente a "No", então iremos colapsar as duas opções para diminuir a cardinalidade.
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in service_cols:
    df[col] = df[col].replace('No internet service', 'No')

# MultipleLines também tem "No phone service" que receberá o mesmo tratamento
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

print(df[service_cols].nunique())

OnlineSecurity      2
OnlineBackup        2
DeviceProtection    2
TechSupport         2
StreamingTV         2
StreamingMovies     2
dtype: int64


## 2. Engenharia de Features
Criação de 3 features derivadas com base em hipóteses do domínio levantadas no EDA.

In [15]:
# Hipótese 1: cliente que paga muito por pouco tempo tem mais risco de churn
df['ChargePerTenure'] = df['MonthlyCharges'] / (df['tenure'] + 1)

# Hipótese 2: quanto mais serviços contratados, maior o engajamento e menor o churn
df['NumServices'] = df[service_cols].apply(
    lambda row: (row == 'Yes').sum(), axis=1
)

# Hipótese 3: clientes nos primeiros 6 meses tem maior risco de abandono
df['IsNewCustomer'] = (df['tenure'] <= 6).astype(int)

print(df[['ChargePerTenure', 'NumServices', 'IsNewCustomer']].describe().round(2))

       ChargePerTenure  NumServices  IsNewCustomer
count          7043.00      7043.00        7043.00
mean              5.77         2.04           0.21
std               8.72         1.85           0.41
min               0.26         0.00           0.00
25%               1.25         0.00           0.00
50%               2.08         2.00           0.00
75%               5.95         3.00           0.00
max              80.85         6.00           1.00


## 3. Encoding, Scaling e Pipeline
Uso de `ColumnTransformer` para aplicar transformações distintas por tipo de feature.  
**Regra fundamental:** `fit_transform` apenas no conjunto de treino e `transform` no teste.

In [16]:
X = df.drop(columns=['Churn'])
y = df['Churn']

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges',
            'ChargePerTenure', 'NumServices']

bin_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
            'PhoneService', 'MultipleLines', 'PaperlessBilling',
            'IsNewCustomer'] + service_cols

cat_cols = ['InternetService', 'Contract', 'PaymentMethod']

print(f"Numéricas: {len(num_cols)} | Binárias: {len(bin_cols)} | Categóricas: {len(cat_cols)}")

Numéricas: 5 | Binárias: 14 | Categóricas: 3


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Churn rate train: {y_train.mean():.3f} | test: {y_test.mean():.3f}")
# Proporção preservada nos dois conjuntos estratificação funcionando perfeitamente.

Train: (5634, 22) | Test: (1409, 22)
Churn rate train: 0.265 | test: 0.265


In [18]:
# Binárias Yes/No -> 0/1 via OrdinalEncoder
binary_transformer = OrdinalEncoder(categories='auto')

# Categóricas nominais → One-Hot (drop='first' evita multicolinearidade)
categorical_transformer = Pipeline([
    ('onehot', OneHotEncoder(drop='first', sparse_output=False))
])

# Numéricas → StandardScaler (média 0, desvio 1)
numeric_transformer = Pipeline([
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('bin', binary_transformer, bin_cols),
    ('cat', categorical_transformer, cat_cols)
], remainder='drop')

# fit_transform no treino, o scaler aprende média e desviosomente do treino
# transform no teste, aplica os mesmos parâmetros sem vazamento de informação
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

print(f"Shape processado — train: {X_train_proc.shape} | test: {X_test_proc.shape}")

Shape processado — train: (5634, 26) | test: (1409, 26)


## 4. Salvar arquivos

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# preprocessor.pkl será carregado no Streamlit (Dia 5) para inferência em produção
joblib.dump(preprocessor, '../models/preprocessor.pkl')
print("Arquivos salvos com sucesso.")

Artefatos salvos com sucesso.


## 5. Decisões e próximos passos

**Decisões tomadas:**
- 11 nulos em `TotalCharges` imputados com mediana por grupo de `Contract` (preserva distribuição por segmento)
- `No internet service` e `No phone service` colapsados para `No` (reduz cardinalidade sem perda de informação)
- 3 features novas criadas com base em hipóteses do domínio: `ChargePerTenure`, `NumServices`, `IsNewCustomer`
- `ColumnTransformer` garante ausência de data leakage
- Split 80/20 estratificado mantém proporção 73/27 de churn em ambos os conjuntos

**Próximo passo (Dia 3):** Treinar e comparar Logistic Regression, Random Forest, XGBoost e LightGBM com cross-validation e tracking de experimentos via MLflow.